# Model training — reviewer-feedback response, part 16 (the thrust-ablation control, revisited for naive(thrust)-only correction)

A direct, important catch: every model built in Parts 13-15 uses `input_names_full`, which
includes `thrust` and `thrust_angle` as ANN inputs - the same six kinematic quantities the
*original* (convex-opt-based) pipeline used. Part 1 tested this exact circularity concern for
the original pipeline (the thrust-ablated model there scored 4.95 deg vs. the full model's
5.55 deg - removing thrust did not hurt, addressing Reviewer #1's circularity concern about
`convex_opt_heading_correction`'s thrust-alignment term). But Part 13 changed the correction
method itself to `naive(thrust)-only` - a correction that is now *more*, not less, tied to
`thrust_angle` (it is literally the sole reference for the global orientation decision, not one
weighted term in an optimization). That control was never re-run for this new correction
method, and every conclusion in Parts 14-15 (the "recovered trajectories cost 0.91 degrees"
finding in particular) was built entirely on thrust-including models. This notebook redoes
the ablation control for the current correction method, and checks whether Part 15's central
finding survives.

## 1. Retrain the FULL (mixed-population) and OVERLAP-ONLY models with thrust ablated

In [1]:
import os
import sys
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../utils'))
from utils import pull_out_individual_trajectories, augment_with_time_delay_embedding, create_model, circular_distance
import keras
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats as scipy_stats

RESPONSE_DIR = '../pipelinedata/reviewer_response'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

old_filtered_ids = set(pd.read_csv('../pipelinedata/04_filtered/all_wind_heading_and_trajectories_augmented_corrected_filtered.csv')['trajec_objid'].unique())
new_fly_data = pd.read_csv(f'{RESPONSE_DIR}/06_final_thrust_naive_only.csv')

TIME_WINDOW = 4
input_names_ablated = ['groundspeed', 'groundspeed_angle', 'airspeed', 'airspeed_angle']
n_input_ablated = len(input_names_ablated) * TIME_WINDOW
n_output = 2
KEEP_COLS = ['heading_angle_x', 'heading_angle_y', 'trajec_objid']

def embed(trajectories, input_names, time_window, wind_augment=False):
    kwargs = dict(time_window=time_window, input_names=input_names, output_names=KEEP_COLS, direction='backward')
    return augment_with_time_delay_embedding(trajectories, wind_augment=wind_augment, wrap_angles=wind_augment, **kwargs)

def angular_error_deg(y_true_xy, y_pred_xy):
    true_angle = np.arctan2(y_true_xy[:, 1], y_true_xy[:, 0])
    pred_angle = np.arctan2(y_pred_xy[:, 1], y_pred_xy[:, 0])
    return np.degrees(circular_distance(true_angle, pred_angle))

def per_trajectory_errors(trajec_id_values, err_deg):
    return (pd.DataFrame({'trajec_objid': trajec_id_values, 'err': err_deg})
            .groupby('trajec_objid')['err'].median())

def grouped_inner_val_split(X, y, groups, val_fraction=0.2, seed=123):
    unique_groups = groups.unique()
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    dummy = np.zeros((len(unique_groups), 1))
    subtrain_gidx, val_gidx = next(gss2.split(dummy, groups=unique_groups))
    subtrain_ids = set(unique_groups[subtrain_gidx])
    val_ids = set(unique_groups[val_gidx])
    m_sub = groups.isin(subtrain_ids)
    m_val = groups.isin(val_ids)
    return (X[m_sub].values, y[m_sub].values), (X[m_val].values, y[m_val].values)

def bootstrap_ci(values, n_boot=5000, ci=95, seed=0):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    boots = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_medians = np.median(boots, axis=1)
    lo, hi = np.percentile(boot_medians, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return float(np.median(values)), float(lo), float(hi)

# FULL (mixed population, includes recovered trajectories)
fly_traj_list = pull_out_individual_trajectories(new_fly_data, min_traj_length=12)
trajec_ids = np.array([t['trajec_objid'].iloc[0] for t in fly_traj_list])
gss_full = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=RANDOM_SEED)
dummy_X = np.zeros((len(fly_traj_list), 1))
train_idx, test_idx = next(gss_full.split(dummy_X, groups=trajec_ids))
train_trajectories_full = [fly_traj_list[i] for i in train_idx]
test_trajectories_full = [fly_traj_list[i] for i in test_idx]
full_test_ids = set(trajec_ids[test_idx])

train_orig_full = embed(train_trajectories_full, input_names_ablated, TIME_WINDOW, wind_augment=False)
train_wind_full = embed(train_trajectories_full, input_names_ablated, TIME_WINDOW, wind_augment=True)
train_all_full = pd.concat([train_orig_full, train_wind_full], ignore_index=True)
X_train_full = train_all_full.iloc[:, 0:n_input_ablated].reset_index(drop=True)
y_train_full = train_all_full[['heading_angle_x', 'heading_angle_y']].reset_index(drop=True)
groups_train_full = train_all_full['trajec_objid'].reset_index(drop=True)

keras.utils.set_random_seed(RANDOM_SEED)
(Xtr, ytr), (Xval, yval) = grouped_inner_val_split(X_train_full, y_train_full, groups_train_full)
model_full_ablated = create_model(n_input=n_input_ablated, n_output=n_output, neurons=20, layers=2)
model_full_ablated.fit(Xtr, ytr, validation_data=(Xval, yval), epochs=150, batch_size=256, verbose=0)
model_full_ablated.save('../models/model_reviewer_response_thrust_naive_only_ablated.keras')

# OVERLAP-ONLY (no recovered trajectories at all)
overlap_only_final = new_fly_data[new_fly_data['trajec_objid'].isin(old_filtered_ids)].copy()
fly_traj_list_overlap = pull_out_individual_trajectories(overlap_only_final, min_traj_length=12)
trajec_ids_overlap = np.array([t['trajec_objid'].iloc[0] for t in fly_traj_list_overlap])
gss_overlap = GroupShuffleSplit(n_splits=1, test_size=0.33, random_state=RANDOM_SEED)
dummy_o = np.zeros((len(fly_traj_list_overlap), 1))
train_idx_o, test_idx_o = next(gss_overlap.split(dummy_o, groups=trajec_ids_overlap))
train_trajectories_o = [fly_traj_list_overlap[i] for i in train_idx_o]
overlap_test_ids = set(trajec_ids_overlap[test_idx_o])

train_orig_o = embed(train_trajectories_o, input_names_ablated, TIME_WINDOW, wind_augment=False)
train_wind_o = embed(train_trajectories_o, input_names_ablated, TIME_WINDOW, wind_augment=True)
train_all_o = pd.concat([train_orig_o, train_wind_o], ignore_index=True)
X_train_o = train_all_o.iloc[:, 0:n_input_ablated].reset_index(drop=True)
y_train_o = train_all_o[['heading_angle_x', 'heading_angle_y']].reset_index(drop=True)
groups_train_o = train_all_o['trajec_objid'].reset_index(drop=True)

keras.utils.set_random_seed(RANDOM_SEED)
(Xtr_o, ytr_o), (Xval_o, yval_o) = grouped_inner_val_split(X_train_o, y_train_o, groups_train_o)
model_overlap_ablated = create_model(n_input=n_input_ablated, n_output=n_output, neurons=20, layers=2)
model_overlap_ablated.fit(Xtr_o, ytr_o, validation_data=(Xval_o, yval_o), epochs=150, batch_size=256, verbose=0)
model_overlap_ablated.save('../models/model_reviewer_response_overlap_only_ablated.keras')

print("Both thrust-ablated models trained.")

I0000 00:00:1789357342.854116 2913591 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789357342.854930 2913591 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1789357342.908264 2913591 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1789357344.247749 2913591 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789357344.248152 2913591 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Detected 3420 trajectories.


3338 trajectories with more than 12 timesteps.


/home/nehal/.pyenv/versions/3.12.12/envs/drosophila_body_orientation_predictor/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1789357370.628247 2913591 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Detected 2386 trajectories.
2340 trajectories with more than 12 timesteps.


/home/nehal/.pyenv/versions/3.12.12/envs/drosophila_body_orientation_predictor/lib/python3.12/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Both thrust-ablated models trained.


## 2. Own-test-set accuracy: does ablating thrust help or hurt, compared to the thrust-including models?

In [2]:
test_embedded_full = embed(test_trajectories_full, input_names_ablated, TIME_WINDOW, wind_augment=False)
X_test_full = test_embedded_full.iloc[:, 0:n_input_ablated].values
y_test_full = test_embedded_full[['heading_angle_x', 'heading_angle_y']].values
pred_test_full = model_full_ablated.predict(X_test_full, verbose=0)
per_traj_test_full = per_trajectory_errors(test_embedded_full['trajec_objid'].values,
                                            angular_error_deg(y_test_full, pred_test_full))
med, lo, hi = bootstrap_ci(per_traj_test_full.values)
print(f"FULL (mixed, thrust-ABLATED) model, own test set (n={len(per_traj_test_full)}): "
      f"{med:.2f} deg [{lo:.2f}, {hi:.2f}]")
print(f"  (thrust-INCLUDING full model, same population, Part 14: 8.55 deg [8.15, 9.13])")

FULL (mixed, thrust-ABLATED) model, own test set (n=1097): 6.94 deg [6.43, 7.33]
  (thrust-INCLUDING full model, same population, Part 14: 8.55 deg [8.15, 9.13])


**Significance:** Ablating thrust IMPROVES the mixed-population model too (6.94 deg vs. 8.55 deg) - consistent with Part 1's original finding on the old correction method (thrust-ablated 4.95 deg vs. full 5.55 deg). Thrust is not merely unnecessary; a model without it is measurably better here, on both correction methods tested so far.

## 3. The decisive test: does the 0.91 deg training cost (Part 15) survive without thrust as an input?

In [3]:
common_test_ids = full_test_ids & overlap_test_ids
print(f"Common held-out trajectories: {len(common_test_ids)} (matches Part 15's 262)")

common_trajectories = [t for t in test_trajectories_full if t['trajec_objid'].iloc[0] in common_test_ids]
common_embedded = embed(common_trajectories, input_names_ablated, TIME_WINDOW, wind_augment=False)
X_common = common_embedded.iloc[:, 0:n_input_ablated].values
y_common_true = common_embedded[['heading_angle_x', 'heading_angle_y']].values

pred_full_common = model_full_ablated.predict(X_common, verbose=0)
pred_overlap_common = model_overlap_ablated.predict(X_common, verbose=0)
per_traj_full_common = per_trajectory_errors(common_embedded['trajec_objid'].values, angular_error_deg(y_common_true, pred_full_common))
per_traj_overlap_common = per_trajectory_errors(common_embedded['trajec_objid'].values, angular_error_deg(y_common_true, pred_overlap_common))

common_ids_sorted = sorted(common_test_ids)
full_vals = per_traj_full_common.reindex(common_ids_sorted).values
overlap_vals = per_traj_overlap_common.reindex(common_ids_sorted).values
diff = full_vals - overlap_vals
med_diff, lo_diff, hi_diff = bootstrap_ci(diff)
wstat, wp = scipy_stats.wilcoxon(full_vals, overlap_vals)
pct = 100 * (np.median(full_vals) - np.median(overlap_vals)) / np.median(overlap_vals)

print(f"\n=== THRUST-ABLATED fair comparison, same 262 common held-out trajectories as Part 15 ===")
print(f"FULL (mixed, ablated):    {np.median(full_vals):.2f} deg")
print(f"OVERLAP-ONLY (ablated):   {np.median(overlap_vals):.2f} deg")
print(f"Paired difference: {med_diff:.2f} deg [{lo_diff:.2f}, {hi_diff:.2f}], "
      f"Wilcoxon p={wp:.2e}, relative {pct:+.1f}%")
print()
print(f"(thrust-INCLUDING models, Part 15, same 262 trajectories: full 7.36 deg, overlap-only "
      f"5.61 deg, paired diff +0.91 deg [0.44, 1.49], p=1.3e-4, relative +31.2%)")

Common held-out trajectories: 262 (matches Part 15's 262)



=== THRUST-ABLATED fair comparison, same 262 common held-out trajectories as Part 15 ===
FULL (mixed, ablated):    5.77 deg
OVERLAP-ONLY (ablated):   5.95 deg
Paired difference: 0.03 deg [-0.60, 0.57], Wilcoxon p=7.15e-01, relative -3.2%

(thrust-INCLUDING models, Part 15, same 262 trajectories: full 7.36 deg, overlap-only 5.61 deg, paired diff +0.91 deg [0.44, 1.49], p=1.3e-4, relative +31.2%)


**Significance:** The 0.91 deg statistically-robust training cost from Part 15 VANISHES once thrust is removed from the model's inputs: 5.77 vs. 5.95 deg, a paired difference of 0.03 deg [95% CI -0.60, 0.57], Wilcoxon p=0.72 - not distinguishable from zero, and the point estimate even flips direction. Part 15's finding was correct as far as it went (the thrust-including model really is worse when trained with the recovered trajectories), but it was a property of that specific architecture, not a general property of the recovered trajectories themselves. This is exactly the ablation control Part 1 ran for the original correction method and that should have been re-run immediately when Part 13 changed the correction to depend on thrust_angle alone - a gap in this response's own methodology, now closed.

## 4. Does the recovered-vs-overlap DIFFICULTY gap survive, even though the training COST does not?

In [4]:
recovered_test_ids = full_test_ids - old_filtered_ids
overlap_test_ids_in_full = full_test_ids & old_filtered_ids

pred_full_own = model_full_ablated.predict(X_test_full, verbose=0)
err_full_own = angular_error_deg(y_test_full, pred_full_own)
per_traj_full_own = per_trajectory_errors(test_embedded_full['trajec_objid'].values, err_full_own)

print(f"THRUST-ABLATED full model, split by trajectory provenance:")
print(f"  Overlap test trajectories   (n={len(overlap_test_ids_in_full)}): "
      f"{per_traj_full_own.loc[list(overlap_test_ids_in_full)].median():.2f} deg")
print(f"  Recovered test trajectories (n={len(recovered_test_ids)}): "
      f"{per_traj_full_own.loc[list(recovered_test_ids)].median():.2f} deg")
print(f"  (thrust-INCLUDING full model, Part 14 Section 6b, same split: overlap 7.02 deg, recovered 21.27 deg)")

THRUST-ABLATED full model, split by trajectory provenance:
  Overlap test trajectories   (n=795): 5.53 deg
  Recovered test trajectories (n=302): 17.62 deg
  (thrust-INCLUDING full model, Part 14 Section 6b, same split: overlap 7.02 deg, recovered 21.27 deg)


**Significance:** The recovered trajectories are STILL substantially harder to predict even with thrust completely absent from the model's inputs (17.62 vs. 5.53 deg, a >3x gap, close to the thrust-including model's 21.27 vs. 7.02 deg). This is the key piece that separates the two mechanisms proposed in Part 15: Mechanism 1 (thrust_angle as a noisy model INPUT) cannot be responsible for this gap, since the ablated model never sees thrust_angle at all - yet the gap persists almost as large. Mechanism 2 (the task itself is less determined at low thrust - a near-hovering fly is not aerodynamically obligated to align its body with a near-zero thrust vector) survives as the explanation for why recovered trajectories are hard, independent of architecture. Mechanism 1 remains the best explanation for why mixing them into TRAINING hurt the thrust-including model on the EASY trajectories too (Section 3) - and that specific harm disappears once the noisy input channel is removed.

## 5. Summary

This notebook re-runs the thrust-ablation control that Part 1 originally used to address
Reviewer #1's circularity concern, for the naive(thrust)-only correction method introduced in
Part 13 - a control that had not yet been re-checked despite the correction now being MORE
tied to thrust_angle (it is the sole reference for the global orientation decision) than the
original convex-opt method was.

**The result revises Part 15's central recommendation.** With thrust ablated from the model's
inputs:

1. The mixed-population (full) model improves overall (6.94 vs. 8.55 deg) - ablating thrust
   is not merely safe, it measurably helps, consistent with Part 1's finding on the original
   correction method.
2. The 0.91 deg "cost" of training with the recovered trajectories, found to be statistically
   robust in Part 15 (Wilcoxon p=1.3e-4), COMPLETELY DISAPPEARS (0.03 deg, p=0.72, direction
   not even consistent). That cost was specific to a model architecture that takes
   thrust_angle - itself noisier at low thrust (Part 15 Section 2b, Mechanism 1) - as an input.
3. The recovered trajectories remain intrinsically harder to predict on their own terms
   (17.62 vs. 5.53 deg) even without thrust as an input - confirming Mechanism 2 (weaker
   aerodynamic coupling between heading and kinematics at low thrust) as a real, architecture-
   independent phenomenon, while Mechanism 1 (noisy input feature) is now isolated as the
   specific cause of the training-cost effect that vanishes here.

**Revised recommendation for the response letter:** for the thrust-ablated architecture (which
Part 1 already preferred on accuracy and circularity grounds, and which this notebook confirms
performs at least as well under the corrected pipeline too), there is no accuracy reason to
exclude the recovered near-hover trajectories from training - Part 15's thrust-magnitude
filtering criterion was a reasonable response to a real finding, but that finding was
architecture-specific and does not generalize to the model this response should actually be
recommending. The remaining, honest consideration is not "should this data be thrown away" but
"the near-hover regime is intrinsically harder to predict, and that should be disclosed
transparently (e.g. reported per-regime, as in Part 3/11) rather than either silently excluded
or silently averaged away in a single pooled number."